Variational Inference: A Review for Statisticiansに乗っている、混合ガウス分布のパラメータ推定を行う。
事前分布やアルゴリズムは、mdファイルに記載

In [143]:
import numpy as np
np.random.seed(42)  # 乱数のシードを固定

In [150]:
x=np.zeros((5,50))
for i in range(5):
    x[i]=np.random.normal(loc=1+i, scale=1, size=50)


In [145]:
x[:,1]

array([0.93086785, 1.80745886, 2.78967734, 4.1732241 , 5.28039226])

In [151]:
class cavi_gmm:
    def __init__(self, k, max_iter=30):
        self.k = k
        self.max_iter = max_iter 

    def fit(self, x):
        x = x.flatten()
        N = len(x)
        mu = np.random.normal(0, 1, self.k) 
        sigma2 = np.ones(self.k)            
        fai = np.zeros([N, self.k])        

        for _ in range(self.max_iter):
            for i in range(N):
                for k in range(self.k):
                    exponent = mu[k] * x[i] - (sigma2[k] + mu[k]**2) / 2.0
                    fai[i, k] = np.exp(exponent)
                fai[i, :] = fai[i, :] / np.sum(fai[i, :])
            for k in range(self.k):
                sum_fai_k = np.sum(fai[:, k])
                denominator = 1 + sum_fai_k 
                numerator = np.sum(fai[:, k] * x) 
                mu[k] = numerator / denominator
                sigma2[k] = 1.0 / denominator
        
        return mu, sigma2, fai

In [152]:
cavi = cavi_gmm(k=5, max_iter=100)
mu, sigma2, fai = cavi.fit(x)
print("推定された平均:", mu)
print("推定された分散:", sigma2) 

推定された平均: [3.74124814 2.36911789 1.03141855 4.91986491 2.19346779]
推定された分散: [0.01881107 0.02035028 0.02164594 0.01730822 0.02052276]


In [153]:
predicted_labels = np.argmax(fai, axis=1)
predicted_labels=predicted_labels.reshape(5, 50)
print("予測されたクラスタラベル:\n", predicted_labels)


予測されたクラスタラベル:
 [[2 4 2 2 4 2 2 2 2 2 2 1 2 2 2 1 2 4 2 2 2 2 2 2 2 2 4 1 2 0 2 2 2 2 2 2
  2 2 4 2 2 4 2 4 4 2 2 4 2 2]
 [4 0 2 1 4 4 0 1 1 0 4 1 4 1 4 4 1 2 0 2 2 0 1 1 1 4 2 4 2 1 4 2 4 1 2 2
  4 4 2 2 4 2 2 2 4 1 0 1 4 4]
 [4 1 1 0 4 0 3 1 0 0 1 0 1 0 4 1 0 3 0 3 4 0 0 3 4 4 1 2 1 4 0 0 3 0 1 4
  0 4 3 0 1 2 3 1 0 2 1 1 1 1]
 [3 1 0 0 3 3 1 1 3 3 0 3 0 3 0 3 3 0 3 3 3 1 3 3 4 1 4 0 3 3 0 3 1 1 0 3
  0 4 0 1 3 3 0 0 1 0 3 1 3 0]
 [0 3 0 3 0 3 3 0 3 3 3 3 3 3 3 3 1 0 3 3 3 3 3 3 3 3 0 0 3 0 0 3 0 3 3 3
  3 3 0 3 3 0 3 0 0 3 3 0 3 3]]


正解率

In [155]:
mapped_labels = np.array([2, 4, 1, 0, 3])
y_true_2d = np.repeat(mapped_labels, 50).reshape(5, 50)
accuracy = np.mean(predicted_labels == y_true_2d)
print("クラスタリングの精度:", accuracy)

クラスタリングの精度: 0.472
